In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import seaborn as sns

%matplotlib inline

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 500) 
pd.set_option('display.expand_frame_repr', False)


# Dataset Exploration

In [ ]:
df = pd.read_csv("../data/raw/UNSW_NB15_training-set.csv")
df.info()

In [ ]:
df

First check for duplicates

In [ ]:
n_before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'id']).reset_index(drop=True)
print(f"Duplicate rows removed: {n_before - len(df)}")
print(f"Remaining rows: {len(df)}")

In [ ]:
df['label'].value_counts()

This dataset is almost balanced, with 51.8% of training data being labelled as attack (1) vs 48.2% being labelled as normal (0).

Correlation Heatmap Analysis, excluding irrelevant columns (id, label, stcpb, dtcpb) 

In [ ]:
num_cols_eda = df.select_dtypes(include='number').columns.difference(['id', 'label', 'stcpb', 'dtcpb']).tolist()

corr_matrix = df[num_cols_eda].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, row, round(upper.loc[row, col], 3))
             for col in upper.columns for row in upper.index
             if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.9]
high_corr.sort(key=lambda x: x[2], reverse=True)

print(f"Feature pairs with correlation > 0.9: {len(high_corr)}")
for a, b, v in high_corr:
    print(f"  {a:30s} <-> {b:30s}  {v}")

In [ ]:
if high_corr:
    corr_features = list(set([f for a, b, _ in high_corr for f in (a, b)]))
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(df[corr_features].corr(), annot=True, fmt='.2f', cmap='coolwarm',
                center=0, ax=ax, linewidths=0.5)
    ax.set_title('Highly Correlated Features (pairs > 0.9)')
    plt.tight_layout()
    plt.show()
else:
    print("No feature pairs with correlation > 0.9")

## Feature Engineering
Let's conduct an EDA on our features to determine any pre-processing that needs to be done within our dataset, as well as if there are any features to exclude or modify to train our models.

First, drop the irrelevant columns being 'id' and 'attack_cat' as we do not need the id nor are we classifying the attack type (a different problem). Additionally, drop the columns 'stcpb' and 'dtcpb' as they are random TCP sequence numbers and will not contribute to the predictions.

In [ ]:
df.drop(['id', 'attack_cat', 'stcpb', 'dtcpb'],axis=1,inplace=True)

Now get categorical vs numerical features

In [ ]:
cat_cols = df.select_dtypes(include=['str']).columns.tolist()
num_cols = df.select_dtypes(include='number').columns.difference(['label']).tolist()

print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"Numerical features  ({len(num_cols)}): {num_cols}")

### Categorical Features — Unique counts & distributions

In [ ]:
for col in cat_cols:
    vc = df[col].value_counts()
    print(f"\n{col}  ({vc.shape[0]} unique values)")
    print(vc.to_string())

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 5))

for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts()
    # Cap at top 15 as proto has 131 values
    if len(vc) > 15:
        vc = vc.head(15)
        ax.set_title(f'{col}\n(top 15 of {df[col].nunique()})')
    else:
        ax.set_title(f'{col}\n({len(vc)} unique)')
    vc.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Categorical Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Numerical Features

In [ ]:
stats = df[num_cols].describe().T[['min', 'max', 'mean', 'std']]
stats['range'] = stats['max'] - stats['min']
stats.sort_values('range', ascending=False).style.format({
    'min':   '{:,.2f}',
    'max':   '{:,.2f}',
    'mean':  '{:,.2f}',
    'std':   '{:,.2f}',
    'range': '{:,.2f}',
})

In [ ]:
skewed_cols = [
    'sload', 'dload', 'sbytes', 'dbytes', 'sjit', 'djit',
    'rate', 'sinpkt', 'dinpkt', 'spkts', 'dpkts',
    'sloss', 'dloss', 'response_body_len'
]

n = len(skewed_cols)
fig, axes = plt.subplots(n, 2, figsize=(12, n * 2.5))

for i, col in enumerate(skewed_cols):
    raw = df[col].dropna()
    log = np.log1p(raw)

    axes[i, 0].hist(raw, bins=60, color='steelblue', edgecolor='none')
    axes[i, 0].set_title(f'{col} — raw', fontsize=9)
    axes[i, 0].set_ylabel('Count', fontsize=8)
    axes[i, 0].tick_params(labelsize=7)

    axes[i, 1].hist(log, bins=60, color='darkorange', edgecolor='none')
    axes[i, 1].set_title(f'{col} — log1p', fontsize=9)
    axes[i, 1].tick_params(labelsize=7)

plt.suptitle('Skewed Features: Raw vs log1p Distributions', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Preprocessing

### Categorical Variables

We conduct preprocessing based on the results of the EDA above. There are three categorical variables: proto, service, and state. For proto, we can see that only 6 out of its 131 unique categories/values have counts of at least 100 in our dataset of 82 322. To prevent falling under the curse of dimensionality and unnessecary complexity when training our different models, we have made the choice to combine all other categories under one new category named "other", to reduce our unique values from 131 to 7. 

This is the same for the service column, where ...

And for state ..

In [ ]:
from sklearn.preprocessing import StandardScaler

proto_filter   = ['tcp', 'udp', 'unas', 'arp', 'ospf', 'sctp']
service_filter = ['-', 'dns', 'http', 'smtp', 'ftp', 'ftp-data', 'ssh', 'pop3']
state_filter  = ['FIN', 'INT', 'CON', 'REQ']

df['proto_filtered']   = df['proto'].where(df['proto'].isin(proto_filter), other='other')
df['service_filtered'] = df['service'].where(df['service'].isin(service_filter), other='other')
df['state_filtered']   = df['state'].where(df['state'].isin(state_filter), other='other')

proto_ohe   = pd.get_dummies(df['proto_filtered'],   prefix='proto')
service_ohe = pd.get_dummies(df['service_filtered'], prefix='service')
state_ohe   = pd.get_dummies(df['state_filtered'],   prefix='state')

ohe_cols = proto_ohe.columns.tolist() + service_ohe.columns.tolist() + state_ohe.columns.tolist()

df = pd.concat([df, proto_ohe, service_ohe, state_ohe], axis=1)

In [ ]:
df.columns

### Numerical Features

Apply StandardScaler to all numerical columns, with scaled columns being added with a _scaled suffix so originals are preserved.

Additionally, there are multiple columns which are right skewed, as shown from the numerical chart and skewed column plots we displayed earlier. We use a log transformation on these columns to handle the skewness so the distribution appears more normal.

In [ ]:
skewed_cols = [
    'sload', 'dload', 'sbytes', 'dbytes', 'sjit', 'djit',
    'rate', 'sinpkt', 'dinpkt', 'spkts', 'dpkts',
    'sloss', 'dloss', 'response_body_len'
]
non_skewed_cols = [c for c in num_cols if c not in skewed_cols]

for col in skewed_cols:
    df[f'{col}_log'] = np.log1p(df[col])

log_cols = [f'{col}_log' for col in skewed_cols]

scaler = StandardScaler()
cols_to_scale = log_cols + non_skewed_cols
scaled_values = scaler.fit_transform(df[cols_to_scale])
scaled_cols = [f'{c}_scaled' for c in cols_to_scale]
df = pd.concat([df, pd.DataFrame(scaled_values, columns=scaled_cols, index=df.index)], axis=1)

In [ ]:
df

df_full: every column: originals, log-transformed, OHE, scaled, and label  
df_model: for LR & MLP: OHE + log-then-scaled (skewed) + scaled (non-skewed) + label  
df_model_tree: for RF & XGBoost: OHE + raw numerical columns + label (no scaling needed)

In [ ]:
df_full = df.copy()
df_model = df[ohe_cols + scaled_cols + ['label']].copy()
df_model_tree = df[ohe_cols + num_cols + ['label']].copy()

print(f"df_full: {df_full.shape}")
print(f"df_model: {df_model.shape}")
print(f"df_model_tree: {df_model_tree.shape}")

In [ ]:
df_full

In [ ]:
df_model

In [ ]:
df_model.columns

In [ ]:
df_model_tree

In [ ]:
df_model_tree.columns

In [ ]:
df_full.to_csv("../data/processed/training_full.csv", index=False)
df_model.to_csv("../data/processed/training_lr_mlp.csv", index=False)
df_model_tree.to_csv("../data/processed/training_xgb_rf.csv", index=False)

## Preprocessing Pipeline 

Using preprocessing.py

In [ ]:
df_full = pd.read_csv("../data/processed/training_full.csv")
df_model = pd.read_csv("../data/processed/training_lr_mlp.csv")
df_model_tree = pd.read_csv("../data/processed/training_xgb_rf.csv")

In [ ]:
import sys
sys.path.append('../src')

from preprocessing.preprocessing import preprocess_train, preprocess_test, save_artifacts

raw_train = pd.read_csv('../data/raw/UNSW_NB15_training-set.csv')
raw_test  = pd.read_csv('../data/raw/UNSW_NB15_testing-set.csv')

fn_full, fn_model, fn_model_tree, artifacts = preprocess_train(raw_train)

print(f"fn_full       shape: {fn_full.shape}")
print(f"fn_model      shape: {fn_model.shape}")
print(f"fn_model_tree shape: {fn_model_tree.shape}")

In [ ]:
save_artifacts(artifacts, '../data/processed/artifacts.pkl')

test_full, test_model, test_model_tree = preprocess_test(raw_test, artifacts)

test_full.to_csv('../data/processed/test_full.csv', index=False)
test_model.to_csv('../data/processed/test_lr_mlp.csv', index=False)
test_model_tree.to_csv('../data/processed/test_xgb_rf.csv', index=False)

print(f"test_full       shape: {test_full.shape}")
print(f"test_model      shape: {test_model.shape}")
print(f"test_model_tree shape: {test_model_tree.shape}")